# PMAG (Protein Metal Affinity GCN)

This project is aimed at calculating the metal affinity for proteins in various bacteria. It uses the BindingDB database and a Graph Convolutional Network. Each molecule is expressed as a graph where the edges are molecules bonded to each other and each data point is a molecule.

## Processing Data

The .sdf file first has to be turned into a graph with the features, where the graph is a torch_geometric.data.Data object, containing the node and edge tables. The node features are

- element = a one-hot encoding of the element
- hybridization = the angle & shape of the bond the atom prefers
- charge = the formal charge of the atom

The edge features are

- bond type = how strong the bond is
- aromaticity = whether the electrons can change between the atoms

In [14]:
import torch_geometric
from lightning.pytorch.callbacks import early_stopping
# Calculating a feature - replace GetHybridization() with whatever
from rdkit import Chem

def grab_first_valid(mols):
    for mol in mols:
        if mol != None: return mol

path_mols = 'data/train_input.csv'
supplier = Chem.SmilesMolSupplier(path_mols, delimiter=',', smilesColumn=0, nameColumn=-1, titleLine=True)
print(list(supplier[0].GetPropNames()))
print('Loaded')

mol = grab_first_valid(supplier)
Chem.SanitizeMol(mol)

for atom in mol.GetAtoms():
    print(atom.GetHybridization())

['Experimental LgK1']
Loaded


[20:32:57] Unusual charge on atom 1 number of radical electrons set to zero
[20:32:57] Unusual charge on atom 1 number of radical electrons set to zero
[20:32:57] Unusual charge on atom 7 number of radical electrons set to zero
[20:32:57] Unusual charge on atom 4 number of radical electrons set to zero


Min: -3.4, Max: 52.0, Mean: 6.603248764195056


In [ ]:
# Generating one-hot encodings
from rdkit import Chem
from joblib import dump

path_mols = 'data/train_input.csv'
supplier = Chem.SmilesMolSupplier(path_mols, delimiter=',', smilesColumn=0, nameColumn=-1, titleLine=True)

ids = {}

for mol in supplier:
    if mol == None: continue

    for atom in mol.GetAtoms():
        if atom.GetSymbol() in ids:
            pass
        else:
            ids[atom.GetSymbol()] = len(ids) + 1

encodings = {}

for i, id in enumerate(ids.items()):
    vector_encoded = [0] * len(ids)
    vector_encoded[i] = 1
    encodings[id[0]] = vector_encoded

dump(encodings, 'data/one-hot.joblib')

In [ ]:
# Graph creation code - run workers.py instead because of IPython multiprocessing issues
import numpy as np
import torch
from torch_geometric.data import Data
import os
import multiprocessing as mp
from rdkit import Chem
from joblib import load

chunk_size = 200
Chem.SetDefaultPickleProperties(Chem.PropertyPickleOptions.AllProps)

def get_chunks(file_path, chunk_size):
    index = 0
    chunk = []
    supplier = Chem.SmilesMolSupplier(file_path, delimiter=',', smilesColumn=0, nameColumn=-1, titleLine=True)

    for mol in supplier:
        if mol is None: continue
        chunk.append(mol)

        if len(chunk) == chunk_size:
            yield (chunk, index)
            chunk = []
            index += 1

    if chunk: yield chunk, index

encodings = load('./data/one-hot.joblib')

def process(chunk):
    chunk_to_save = []
    lgk1_list = []

    for mol in chunk[0]:
        lgk1 = float(mol.GetProp('Experimental LgK1'))
        lgk1_list.append(lgk1)

    lgk1_list = np.array(lgk1_list)
    lgk1_mean = np.mean(lgk1_list)
    lgk1_std = np.std(lgk1_list)

    for mol in chunk[0]:
        X = []
        edge_features = []

        edge_matrix_idx1 = []
        edge_matrix_idx2 = []

        for atom in mol.GetAtoms():
            X.append([*encodings[atom.GetSymbol()], int(atom.GetHybridization()), atom.GetFormalCharge()])

        for bond in mol.GetBonds():
            features_for_edge = [bond.GetBondType().real, float(bond.GetIsAromatic())]
            edge_features.append(features_for_edge)
            edge_features.append(features_for_edge)

            edge_matrix_idx1.append(bond.GetBeginAtomIdx())
            edge_matrix_idx2.append(bond.GetEndAtomIdx())

            edge_matrix_idx2.append(bond.GetBeginAtomIdx())
            edge_matrix_idx1.append(bond.GetEndAtomIdx())

        edge_matrix = torch.from_numpy(np.array([edge_matrix_idx1, edge_matrix_idx2]))
        X = torch.from_numpy(np.array(X))
        edge_features = torch.from_numpy(np.array(edge_features))

        lgk1 = torch.tensor(float(mol.GetProp('Experimental LgK1')))
        norm_lgk1 = (lgk1 - lgk1_mean) / lgk1_std

        graph = Data(x=X, edge_index=edge_matrix, edge_attr=edge_features, y=norm_lgk1)

        chunk_to_save.append(graph)

    if len(chunk_to_save) == chunk_size:
        torch.save(chunk_to_save, f'./data/graph_chunks/{chunk[1]:04d}.pt')

if __name__ == '__main__':
    mols_path = 'data/train_input.csv'

    with mp.Pool(processes=5) as pool:
        results = pool.imap(process, get_chunks(mols_path, chunk_size))
        for _ in results: pass

## Training the Model

The model is a graph convolutional network with message passing in pytorch.

In [15]:
import os
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

import warnings
warnings.filterwarnings("ignore", ".*improve performance.*")

import lightning as L
import torch, gc
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv, global_add_pool
import torch_geometric

class PMAG(L.LightningModule):
    def __init__(self, node_dim, edge_dim):
        super().__init__()

        layers = []

        for part in range(4):
            if part == 0:
                layers.append(nn.Linear(node_dim, 128))
            else:
                layers.append(nn.Linear(128, 128))

            layers.append(nn.Linear(128, 128))
            layers.append(nn.LayerNorm(128))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.2))

        mlp = nn.Sequential(*layers)
        self.conv = GINEConv(nn=mlp, train_eps=True, edge_dim=edge_dim)
        self.head = nn.Linear(128, 1)

    def forward(self, data):
        X = F.relu(self.conv(data.x, data.edge_index, data.edge_attr))
        embeds = global_add_pool(X, data.batch)
        return self.head(embeds)

    def training_step(self, batch, batch_idx):
        pred = self(batch).squeeze(-1)
        loss = F.mse_loss(pred, batch.y)
        self.log("train_loss", loss.item(), prog_bar=True, on_step=True, on_epoch=True, batch_size=batch.num_graphs)
        return loss

    def validation_step(self, batch, batch_idx):
        pred = self(batch).squeeze(-1)
        loss = F.mse_loss(pred, batch.y)
        self.log("val_loss", loss.item(), prog_bar=True, on_step=True, on_epoch=True, batch_size=batch.num_graphs)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-3, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-5)
        return {"optimizer": optimizer, "lr_scheduler": scheduler}

    def on_epoch_end(self):
        gc.collect()
        if torch.backends.mps.is_available():
            torch.mps.empty_cache()

graphs = []

for filepath in os.listdir('./data/graph_chunks'):
    if filepath.endswith('.pt'):
        chunk = torch.load(os.path.join('./data/graph_chunks', filepath), weights_only=False)

        for graph in chunk:
            graph.x = graph.x.float()
            graph.edge_attr = graph.edge_attr.float()
            graph.edge_index = graph.edge_index.long()

            if isinstance(graph.y, torch.Tensor):
                pass
            else:
                graph.y = torch.tensor(float(graph.y), dtype=torch.float32)

            graphs.append(graph)

num_graphs = len(graphs) * 0.9
train_graphs = graphs[:int(num_graphs)]
val_graphs = graphs[int(num_graphs):]

batch_size = 256
t_loader = torch_geometric.loader.DataLoader(train_graphs, batch_size=batch_size, num_workers=0, shuffle=True)
v_loader = torch_geometric.loader.DataLoader(val_graphs, batch_size=batch_size, num_workers=0, shuffle=False)

checkpoint = L.pytorch.callbacks.ModelCheckpoint(
    dirpath='./models',
    filename='{epoch:02d}-{val_loss:.2f}',
    monitor='val_loss',
    mode='min',
)

early_stopping = L.pytorch.callbacks.EarlyStopping(
    monitor='val_loss',
    min_delta=0.01,
    patience=5,
)

model = PMAG(graphs[0].x.shape[1], graphs[0].edge_attr.shape[1])
trainer = L.Trainer(max_epochs=50, callbacks=[checkpoint, early_stopping], accelerator='mps', devices=1)
trainer.fit(model=model, train_dataloaders=t_loader, val_dataloaders=v_loader)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/Users/zacharypotishko/Desktop/BiochemistryRAG/.venv/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /Users/zacharypotishko/Desktop/PMAG (Protein Metal Affinity GINE)/models exists and is not empty.


┏━━━┳━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ conv │ GINEConv │  127 K │ train │     0 │
│ 1 │ head │ Linear   │    129 │ train │     0 │
└───┴──────┴──────────┴────────┴───────┴───────┘

Trainable params: 128 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 128 K                                                                                                
Total estimated model params size (MB): 0.512                                                                      
Modules in train mode: 25                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/Users/zacharypotishko/Desktop/BiochemistryRAG/.venv/lib/python3.11/site-packages/rich/live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/zacharypotishko/Desktop/BiochemistryRAG/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytr
ee.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and 
treespec.is_leaf()` instead.